# Fitbit: Calorie Burn Prediction & Workout Pattern Clustering
## Project 3

### Problem Statement
Predict calories burned per workout session using supervised regression techniques, and identify hidden workout patterns using unsupervised clustering.

### Business Use Cases
1. **Wearable Fitness Apps:** Real-time calorie burn prediction during workouts.
2. **Personalized Fitness Coaching:** Workout intensity and duration recommendations.
3. **Health Monitoring Platforms:** Energy expenditure insights for nutrition planning.
4. **User Segmentation:** Identifying workout behavior patterns without labels.
5. **Product Optimization:** Feature insights for fitness device manufacturers.

In [ ]:
# ==========================================
# STEP 1: IMPORT NECESSARY LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Algorithms for Task 1 (Supervised Learning - Regression)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Algorithms for Task 2 (Unsupervised Learning - Clustering)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

### Step 2: Load Data
Load the dataset using Pandas. Ensure that 'Fitbit_dataset.csv' is uploaded to your Colab session.

In [ ]:
# ==========================================
# STEP 2: DATA LOADING 
# ==========================================

# 1. Load the dataset
df = pd.read_csv('Fitbit_dataset.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)


### Step 3: Exploratory Data Analysis (EDA)
Addressing the EDA guidelines: Checking data shape, missing values, duplicates, outliers, distributions, and correlations.

In [ ]:
# ==========================================
# STEP 3: EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================

# 1. Basic Characteristics & Structure (Data types, Memory, Rows & Columns)
print("--- Dataset Shape (Rows, Columns) ---")
print(df.shape)

print("\n--- Data Types & Memory Usage ---")
df.info()

# 2. Missing Values & Correctness (Duplicates)
print("\n--- Missing Values Check ---")
print(df.isnull().sum())

print(f"\n--- Duplicate Rows Check ---")
print(f"Total duplicate rows: {df.duplicated().sum()}")

# 3. Variability, Mean, Max, Min (Statistical Summary)
print("\n--- Statistical Summary ---")
display(df.describe())

# 4. Correlation Analysis (Patterns & Relationships)
plt.figure(figsize=(10, 6))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap (Identifying Relationships)")
plt.show()

# 5. Distribution & Outliers Visual Check
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Calories_Burned (kcal)'], kde=True, color='blue')
plt.title('Distribution of Calories Burned')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Calories_Burned (kcal)'], color='orange')
plt.title('Outlier Check for Calories Burned')
plt.show()

### Step 4: Data Preprocessing
Handling missing values and capping outliers based on the Interquartile Range (IQR).

In [ ]:
# ==========================================
# STEP 4: DATA PREPROCESSING
# ==========================================

df_clean = df.copy()

# 1. Handle Missing Values (Median for numbers, Mode for text)
num_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df_clean.select_dtypes(include=['object']).columns

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 2. Outlier Detection and Capping (IQR Method)
for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    
    # Cap logic
    df_clean.loc[df_clean[col] > upper_limit, col] = upper_limit
    df_clean.loc[df_clean[col] < lower_limit, col] = lower_limit

print("Data Preprocessing Complete: Missing values imputed, Outliers capped.")

### Step 5: Task 1 - Supervised Learning (Calorie Burn Prediction)
**Algorithms Used:** Linear Regression, Ridge, Lasso, KNN, Decision Tree, Random Forest, SVR, XGBoost.
**Evaluation Metrics:** MAE, RMSE, R² Score.
**Expected Goal:** Achieve R² ≥ 0.80.
**Additional Requirement:** Hyperparameter Tuning (GridSearchCV).

In [ ]:
# ==========================================
# STEP 5: CALORIE BURN PREDICTION (REGRESSION)
# ALGORITHMS: Linear, Ridge, Lasso, KNN, Decision Tree, Random Forest, SVR, XGBoost
# ADDITIONAL: HYPERPARAMETER TUNING
# EVALUATION METRICS: MAE, RMSE, R2 SCORE
# EXPECTED GOAL: R2 >= 0.80 (Suitable for mobile app deployment)
# ==========================================

target_col = 'Calories_Burned (kcal)'

# 1. Encode categorical variables to numeric & save encoders for Streamlit
df_encoded = df_clean.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# 2. Train/Test Split (80% training, 20% testing)
X = df_encoded.drop(target_col, axis=1)
y = df_encoded[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Feature Scaling & save scaler for Streamlit
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Define Algorithms to Compare
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'KNN Regressor': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'SVR': SVR(),
    'XGBoost': XGBRegressor(random_state=42)
}

results = []

# 5. Train Models and evaluate against metrics
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    
    results.append({'Algorithm': name, 'R2 Score': r2, 'MAE': mae, 'RMSE': rmse})

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
display(results_df)
best_r2 = results_df.iloc[0]['R2 Score']
print(f"\nExpected Goal Check: Is R2 >= 0.80? {'YES' if best_r2 >= 0.80 else 'NO'} (Best R2: {best_r2:.4f})")

# ---------------------------------------------------------
# ADDITIONAL REQUIREMENT: HYPERPARAMETER TUNING
# ---------------------------------------------------------
print("\n--- Performing Additional Step: Hyperparameter Tuning (Random Forest) ---")
param_grid = {'n_estimators': [50, 100], 'max_depth': [10, 20, None]}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='r2')
grid_search.fit(X_train_scaled, y_train)
tuned_rf = grid_search.best_estimator_
tuned_r2 = r2_score(y_test, tuned_rf.predict(X_test_scaled))
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Tuned Random Forest R2 Score: {tuned_r2:.4f}")

# 6. Export Models and Transformers as a Pickle file for Streamlit Application
export_data = {
    'scaler': scaler,
    'label_encoders': label_encoders,
    'features': list(X.columns),
    'models': {
        'Linear Regression': models['Linear Regression'],
        'Ridge Regression': models['Ridge Regression'],
        'KNN Regressor': models['KNN Regressor'],
        'Decision Tree': models['Decision Tree'],
        'Random Forest': tuned_rf  # Exporting the hyperparameter tuned model
    }
}
with open('fitbit_models.pkl', 'wb') as f:
    pickle.dump(export_data, f)
print("\nModels and preprocessors successfully saved to 'fitbit_models.pkl' for Streamlit integration!")

### Step 6: Task 2 - Unsupervised Learning (Workout Pattern Clustering)
**Algorithms Used:** PCA (Dimensionality Reduction), K-Means Clustering.
**Additional Algorithms Covered:** Hierarchical Clustering, DBSCAN.
**Evaluation Metric:** Silhouette Score.
**Expected Goal:** Evaluate against the two Silhouette Score Conditions.

In [ ]:
# ==========================================
# STEP 6: WORKOUT PATTERN CLUSTERING
# ALGORITHMS: PCA, KMeans
# ADDITIONAL ALGORITHMS: Hierarchical Clustering, DBSCAN
# EVALUATION METRIC: SILHOUETTE SCORE
# ==========================================

# 1. Drop labels (We don't want to use Workout_Type for unsupervised learning)
df_cluster = df_clean.drop(['Workout_Type', target_col], axis=1, errors='ignore')

# 2. One-Hot Encode categorical variables
df_cluster_encoded = pd.get_dummies(df_cluster, drop_first=True)

# 3. Scale the data
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(df_cluster_encoded)

# 4. Apply PCA Algorithm for Dimensionality Reduction (Variation Step)
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_cluster_scaled)

# 5. Apply Mandatory KMeans Algorithm
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_clusters = kmeans.fit_predict(X_pca)
kmeans_score = silhouette_score(X_pca, kmeans_clusters)
print(f"K-Means Silhouette Score: {kmeans_score:.4f}")

print("\n--- Silhouette Score Evaluation Conditions ---")
print("Condition 1 (> 0.40): Clear behavioral segments.")
print("Condition 2 (0.15 - 0.40): Physiological overlap between workout types.")
if kmeans_score > 0.40:
    print("Result: Condition 1 met. Segments are clearly separated.")
elif kmeans_score >= 0.15:
    print("Result: Condition 2 met. Score is valid. There is physiological overlap between workout types.")
else:
    print("Result: Score too low (< 0.15).")

# ---------------------------------------------------------
# ADDITIONAL REQUIREMENT: HIERARCHICAL CLUSTERING & DBSCAN
# ---------------------------------------------------------
print("\n--- Additional Clustering Algorithms ---")
# Apply Hierarchical Clustering (Agglomerative)
hierarchical = AgglomerativeClustering(n_clusters=4)
h_clusters = hierarchical.fit_predict(X_pca)
h_score = silhouette_score(X_pca, h_clusters)
print(f"Hierarchical Silhouette Score: {h_score:.4f}")

# Apply DBSCAN (Density-Based Spatial Clustering)
dbscan = DBSCAN(eps=2.5, min_samples=5)
d_clusters = dbscan.fit_predict(X_pca)
if len(set(d_clusters)) > 1:
    d_score = silhouette_score(X_pca, d_clusters)
    print(f"DBSCAN Silhouette Score: {d_score:.4f}")
else:
    print("DBSCAN found only 1 continuous cluster or noise.")
# ---------------------------------------------------------

# 6. Save K-Means Clusters back to dataset for Business Interpretation
df_clean['Cluster'] = kmeans_clusters

# Show Business Insight (How clusters map to Original Workout Type)
if 'Workout_Type' in df.columns:
    print("\n--- Business Insight: Hidden Clusters mapped to Original Workout Types ---")
    display(pd.crosstab(df_clean['Cluster'], df['Workout_Type']))